# Create James N. Kirby Foundation Awards

Creates OpenAlex award rows from the James N. Kirby Foundation's official Recent Grants table.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/kirby_to_s3.py` to upload the parquet file.
- Contractor local validation used `python3 scripts/local/kirby_to_s3.py --skip-upload`.

**Source authority:** first-party James N. Kirby Foundation WordPress page/API:
- `https://www.kirbyfoundation.com.au/grants/recent-grants/`
- `https://www.kirbyfoundation.com.au/wp-json/wp/v2/pages?slug=recent-grants`

**S3 input:** `s3a://openalex-ingest/awards/kirby/kirby_recent_grants.parquet`

**Funder:** James N. Kirby Foundation, OpenAlex `F4320314616`, country `AU`, no DOI/ROR in OpenAlex as of Step 0.

**Coverage note:** the public Recent Grants table currently covers 2025 and 2024 only. It publishes recipient/project rows with amount columns for each recent year; this notebook models one award row per non-empty `(recipient/project, year amount)` cell. Currency is `AUD`, implicit from the Australian funder/source context. Recipient country is left NULL because the table does not expose per-recipient country.

**Provenance:** `kirby_recent_grants`


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kirby_recent_grants_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/kirby/kirby_recent_grants.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 216 rows.
SELECT COUNT(*) as total_kirby_recent_grants
FROM openalex.awards.kirby_recent_grants_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.kirby_recent_grants_raw;


In [ ]:
%sql
-- Sample raw Kirby data.
SELECT
    funder_award_id,
    display_name,
    recipient_name,
    source_category,
    source_year,
    amount,
    currency,
    source_amount_display,
    landing_page_url
FROM openalex.awards.kirby_recent_grants_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'kirby_recent_grants_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(MIN(TRY_CAST(amount AS DOUBLE)), 0) AS min_amount,
    ROUND(PERCENTILE_APPROX(TRY_CAST(amount AS DOUBLE), 0.5), 0) AS median_amount,
    ROUND(MAX(TRY_CAST(amount AS DOUBLE)), 0) AS max_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_aud,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies,
    COUNT(source_year) AS rows_with_year
FROM openalex.awards.kirby_recent_grants_raw;


In [ ]:
%sql
-- Native-key inspection. funder_award_id is synthetic from source year, category, recipient, and description hash.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT CONCAT(source_year, ':', source_category, ':', recipient_name, ':', COALESCE(description, ''))) AS distinct_source_rows
FROM openalex.awards.kirby_recent_grants_raw;


In [ ]:
%sql
-- Source year/category distribution.
SELECT
    source_year,
    source_category,
    COUNT(*) AS cnt,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_aud
FROM openalex.awards.kirby_recent_grants_raw
GROUP BY source_year, source_category
ORDER BY source_year DESC, source_category;


In [ ]:
%sql
-- Description and recipient coverage from the official table.
SELECT
    COUNT(*) AS total,
    COUNT(recipient_name) AS has_recipient_name,
    COUNT(description) AS has_description,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description,
    COUNT(landing_page_url) AS has_landing_page_url
FROM openalex.awards.kirby_recent_grants_raw;


## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this fails, STOP: the funder is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for James N. Kirby Foundation F4320314616'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320314616;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320314616;


## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.kirby_awards
USING delta
AS
WITH
kirby_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314616
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE WHEN TRY_CAST(amount AS DOUBLE) IS NOT NULL THEN 'AUD' ELSE NULL END AS parsed_currency,
        TRY_CAST(source_year AS INT) AS parsed_year,
        CASE
            WHEN TRY_CAST(source_year AS INT) IS NOT NULL
            THEN TRY_TO_DATE(CONCAT(source_year, '-01-01'), 'yyyy-MM-dd')
            ELSE NULL
        END AS parsed_start_date
    FROM openalex.awards.kirby_recent_grants_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'grant' as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'kirby_recent_grants' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        CASE WHEN g.parsed_year > YEAR(current_date()) + 1 THEN NULL ELSE g.parsed_year END as start_year,
        CAST(NULL AS INT) as end_year,

        struct(
            CAST(NULL AS STRING) as given_name,
            CAST(NULL AS STRING) as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.recipient_name), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING,country:STRING,ids:ARRAY<STRUCT<id:STRING,type:STRING,asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING,country:STRING,ids:ARRAY<STRUCT<id:STRING,type:STRING,asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,
        CONCAT('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN kirby_funder f
)

SELECT *
FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 164

In [ ]:
%sql
-- Remove previous Kirby recent-grants data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kirby_recent_grants' AND priority = 164;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    164 as priority
FROM openalex.awards.kirby_awards;


## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_kirby_awards
FROM openalex.awards.kirby_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.kirby_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    lead_investigator.affiliation.name AS recipient_org,
    landing_page_url
FROM openalex.awards.kirby_awards
LIMIT 10;


In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.kirby_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only James N. Kirby Foundation.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.kirby_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.affiliation.name) as has_recipient_org,
    ROUND(SUM(amount), 0) as total_aud,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) as pct_recipient_org
FROM openalex.awards.kirby_awards;


In [ ]:
%sql
-- Amount/currency fail-fast check for this grant pattern.
-- Local source validation found 100% amount coverage and only AUD.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS rows_with_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_positive_amount,
    ROUND(SUM(amount), 0) AS total_amount
FROM openalex.awards.kirby_awards;


In [ ]:
%sql
-- Year and scheme distribution.
SELECT start_year, funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_aud
FROM openalex.awards.kirby_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year DESC, funder_scheme;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 164.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_aud
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'kirby_recent_grants' AND priority = 164
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor has no S3/Databricks access. Admin must:

1. Run `scripts/local/kirby_to_s3.py` without `--skip-upload` to upload `s3://openalex-ingest/awards/kirby/kirby_recent_grants.parquet`.
2. Run this notebook in Databricks.
3. Inspect the Step 6 verification cells, especially row count, uniqueness, amount/currency coverage, and priority 164 insertion.
4. Only add job YAML after the Databricks run and QA are complete.
